# 0. Импорт и конфигурация

In [ ]:
RANDOM_STATE = 42

# from google.colab import drive
# drive.mount('/content/drive')
import os
import math
import json
import numpy as np
import pandas as pd
from typing import List, Iterable, Tuple

def set_global_seed(seed: int = RANDOM_STATE) -> None:
    np.random.seed(seed)

set_global_seed(RANDOM_STATE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 1. Обработка данных

## Шаг 1. Считать данные

In [2]:
Student_ID = 31

In [3]:
datasets = [
    (
        'Give Me Some Credit',
        'https://www.kaggle.com/competitions/GiveMeSomeCredit/overview',
        'SeriousDlqin2yrs'
    ),
    (
        'Porto Seguro’s Safe Driver Prediction',
        'https://www.kaggle.com/competitions/porto-seguro-safe-driver-prediction/overview',
        'target'
    ),
    (
        'Statlog (Shuttle)',
        'https://archive.ics.uci.edu/dataset/148/statlog+shuttle',
        'class'
    ),
    (
        'HTRU2',
        'https://archive.ics.uci.edu/dataset/372/htru2',
        'class'
    ),
    (
        'Bank Marketing',
        'https://archive.ics.uci.edu/dataset/222/bank%2Bmarketing',
        'y'
    ),
]

dataset_id = None if Student_ID is None else Student_ID % len(datasets)
if dataset_id is None:
    print("ОШИБКА! Не указан порядковый номер студента в списке группы.")
else:
    print(f"Информация о датасете '{datasets[dataset_id][0]}' доступна по следующей ссылке: {datasets[dataset_id][1]}")
    print(f"Целевая переменная: {datasets[dataset_id][2]}")

Информация о датасете 'Porto Seguro’s Safe Driver Prediction' доступна по следующей ссылке: https://www.kaggle.com/competitions/porto-seguro-safe-driver-prediction/overview
Целевая переменная: target


Загрузите данные и считайте их в датафрейм

In [ ]:
# df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/train.csv')
df = pd.read_csv('data/train.csv')
df = df.sample(frac=0.05, random_state=42)
df.head()


,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
256886,642026,0,4,1,5,1,0,1,0,0,...,7,3,2,3,0,1,0,1,0,0
118785,297043,0,6,2,10,1,0,0,0,0,...,6,3,3,5,0,1,1,0,0,0
56083,140591,0,4,1,9,1,0,0,0,1,...,3,1,0,7,0,0,1,0,0,0
542002,1354540,0,0,1,7,1,4,0,1,0,...,1,1,3,6,1,1,0,0,0,0
349518,873173,0,1,1,3,1,0,1,0,0,...,6,1,5,6,0,1,0,0,0,0


## Шаг 2. Обработка данных

В обработке датасета вы имеете (почти) полную свободу (важно в итоге просто побить бейзлайн).

Что НЕОБХОДИМО сделать:
- обработать пропуски, если есть
- закодировать категориальные фичи

Что ПОЛЕЗНО сделать:
- удалить дубликаты, если есть
- обработать выбросы
- удалить лишние фичи, если есть (возможно, полезно будет посмотреть на корреляции числовых признаков)
- стандартизировать данные

ВАЖНО: обработанный по необходимым пунктам датафрейм запишите в переменную df_base, обработанный далее по вашему желанию датафрейм запишите в переменную df_processed.

НЕ ПЕРЕЗАПИСЫВАЙТЕ ЭТИ ПЕРЕМЕННЫЕ И df, иначе могут возникнуть проблемы с прохождением тестов

In [6]:
df_base = df.copy().dropna()
df_base.head()

,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
256886,642026,0,4,1,5,1,0,1,0,0,...,7,3,2,3,0,1,0,1,0,0
118785,297043,0,6,2,10,1,0,0,0,0,...,6,3,3,5,0,1,1,0,0,0
56083,140591,0,4,1,9,1,0,0,0,1,...,3,1,0,7,0,0,1,0,0,0
542002,1354540,0,0,1,7,1,4,0,1,0,...,1,1,3,6,1,1,0,0,0,0
349518,873173,0,1,1,3,1,0,1,0,0,...,6,1,5,6,0,1,0,0,0,0


In [7]:
df_processed = df_base.copy().drop_duplicates()

feature_columns = df_processed.columns.drop(['id', 'target'])

for feature in feature_columns:
    df_processed[feature] = (df_processed[feature] - df_processed[feature].mean()) / df_processed[feature].std()

df_processed.head()

,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
256886,642026,0,1.061498,-0.539318,0.222019,1.180046,-0.296998,1.242426,-0.590003,-0.444815,...,0.667972,1.310530,-0.517900,-1.653803,-0.37335,0.770238,-1.110599,1.573357,-0.732522,-0.423992
118785,297043,0,2.071449,0.961163,2.075560,1.180046,-0.296998,-0.804850,-0.590003,-0.444815,...,0.236933,1.310530,0.072057,-0.923315,-0.37335,0.770238,0.900385,-0.635562,-0.732522,-0.423992
56083,140591,0,1.061498,-0.539318,1.704852,1.180046,-0.296998,-0.804850,-0.590003,2.248052,...,-1.056184,-0.365131,-1.697815,-0.192827,-0.37335,-1.298256,0.900385,-0.635562,-0.732522,-0.423992
542002,1354540,0,-0.958403,-0.539318,0.963435,1.180046,2.677578,-0.804850,1.694849,-0.444815,...,-1.918262,-0.365131,0.072057,-0.558071,2.67836,0.770238,-1.110599,-0.635562,-0.732522,-0.423992
349518,873173,0,-0.453427,-0.539318,-0.519398,1.180046,-0.296998,1.242426,-0.590003,-0.444815,...,0.236933,-0.365131,1.251972,-0.558071,-0.37335,0.770238,-1.110599,-0.635562,-0.732522,-0.423992


## Шаг 3. Разделение на train/val/test


Разделите датафрейм на фичи и на целевую переменную.

In [8]:
X_base, y_base = df_base.iloc[:, 2:], df_base['target']
X_processed, y_processed = df_processed.iloc[:, 2:], df_processed['target']

Разделите датафреймы base и processed каждый на выборки train/val/test (запишите их в переменные ниже)

In [9]:
from sklearn.model_selection import train_test_split

X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_base, y_base, test_size=0.2, random_state=42, stratify=y_base
)


In [10]:
# Разделяем на train/test (80/20)
X_train_processed, X_test_processed, y_train_processed, y_test_processed = train_test_split(
    X_processed, y_processed, test_size=0.2, random_state=42, stratify=y_processed
)

# Для валидационной выборки - берем 25% от train (20% от всех данных)
X_train_processed, X_val_processed, y_train_processed, y_val_processed = train_test_split(
    X_train_processed, y_train_processed, test_size=0.25, random_state=42, stratify=y_train_processed
)

## Дополнительное задание*. Решение дисбаланса классов

Изучите ваш датасет на наличие дисбаланса классов. Постройте распределение классов таргет переменной.
Визуализируйте расположение классов через PCA.

Выберите стратегию, по которой будете компенсировать дисбаланс (undersampling, oversampling, интерполяция, генерация новых примеров).
Подсказка: воспользуйтесь библиотекой `imblearn`.
Постройте визуализацию для сбалансированного датасета.

Рекомендуется далее в задании 3 сравнить различные модели на устойчивость к дисбалансу классов. Визуализация в любом виде приветствуется.

In [11]:

### BEGIN YOUR CODE

df_balanced = ...

### END YOUR CODE

# 2. Реализация метрик

Реализуйте метрики классификации с помощью `numpy`/`pandas`. В этом разделе запрещено использовать `sklearn.metrics`.

In [12]:
# noinspection PyUnresolvedReferences,PyTypeChecker
def accuracy_manual(y_true: Iterable[int], y_pred: Iterable[int]) -> float:
    """
    Реализовать accuracy = correct / total.
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    return np.mean(y_true == y_pred)

In [13]:
# noinspection PyUnresolvedReferences,PyTypeChecker
def precision_manual(y_true: Iterable[int], y_pred: Iterable[int]) -> float:
    """
    Реализовать precision = TP / (TP + FP).
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    TP = np.sum((y_true == 1) & (y_pred == 1))
    FP = np.sum((y_true == 0) & (y_pred == 1))

    if TP + FP == 0:
        return 0.0
    return TP / (TP + FP)

In [14]:
# noinspection PyUnresolvedReferences,PyTypeChecker
def recall_manual(y_true: Iterable[int], y_pred: Iterable[int]) -> float:
    """
    Реализовать recall = TP / (TP + FN).
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    TP = np.sum((y_true == 1) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))

    if TP + FN == 0:
        return 0.0
    return TP / (TP + FN)

In [15]:
# noinspection PyUnresolvedReferences,PyTypeChecker
def f1_manual(y_true: Iterable[int], y_pred: Iterable[int]) -> float:
    """
    Реализовать F1 = 2 * P * R / (P + R).
    """
    P = precision_manual(y_true, y_pred)
    R = recall_manual(y_true, y_pred)

    if P + R == 0:
        return 0.0

    return (2 * P * R) / (P + R)

In [16]:
# noinspection PyUnresolvedReferences,PyTypeChecker
def precision_recall_curve_manual(y_true: Iterable[int], y_score: Iterable[float]) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Реализовать подсчет PR-кривой.
    Возвращать (precision_list, recall_list, thresholds) при варьировании порога.
    """
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    thresholds = np.sort(np.unique(y_score))

    precision_list = []
    recall_list = []

    for t in thresholds:
        y_pred = (y_score >= t).astype(int)

        p = precision_manual(y_true, y_pred)
        r = recall_manual(y_true, y_pred)

        precision_list.append(p)
        recall_list.append(r)

    return np.array(precision_list), np.array(recall_list), thresholds


In [17]:
# noinspection PyUnresolvedReferences,PyTypeChecker
def roc_curve_manual(y_true: Iterable[int], y_score: Iterable[float]) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Реализовать подсчет ROC-кривой.
    Возвращать (fpr_list, tpr_list, thresholds).
    """
    thresholds = np.sort(np.unique(y_score))

    fpr_list = []
    tpr_list = []

    for t in thresholds:
        y_pred = (y_score >= t).astype(int)

        tpr = recall_manual(y_true, y_pred)

        FP = np.sum((y_true == 0) & (y_pred == 1))
        TN = np.sum((y_true == 0) & (y_pred == 0))
        fpr = FP / (FP + TN) if (FP + TN) != 0 else 0.0

        fpr_list.append(fpr)
        tpr_list.append(tpr)

    return np.array(fpr_list), np.array(tpr_list), thresholds

In [51]:
# noinspection PyUnresolvedReferences,PyTypeChecker
def roc_auc_manual(fpr: Iterable[float], tpr: Iterable[float]) -> float:
    """
    Реализовать численную интеграцию по FPR (например, трапеции).
    """
    fpr = np.array(fpr)
    tpr = np.array(tpr)

    order = np.argsort(fpr)
    fpr_sorted = fpr[order]
    tpr_sorted = tpr[order]

    auc = np.trapezoid(tpr_sorted, fpr_sorted)
    return float(auc)


# 3. Классификация (sklearn)

Обучите различные алгоритмы классификации и сравните их между собой. В качестве baseline используйте логистическую регрессию.

## Шаг 1. Бейзлайн

In [48]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(
    solver='liblinear',
    penalty='l1',
    class_weight={0: 1, 1: 20},
    C=0.1,
    max_iter=2000,
    random_state=42
)
logreg.fit(X_train_base, y_train_base)

LogisticRegression(C=0.1, class_weight={0: 1, 1: 20}, max_iter=2000,
                   penalty='l1', random_state=42, solver='liblinear')

## Шаг 2. KNN

Воспользуйтесь `sklearn.neighbors import KNeighborsClassifier`. Сравните с бейзлайном.

In [56]:
from sklearn.neighbors import KNeighborsClassifier

neigh = KNeighborsClassifier(
    n_neighbors=7,
    weights='uniform',
    metric='euclidean'
)
neigh.fit(X_train_processed, y_train_processed)

KNeighborsClassifier(metric='euclidean', n_neighbors=7)

## Шаг 3. Решающее дерево

Воспользуйтесь `sklearn.tree import DecisionTreeClassifier`. Сравните с бейзлайном.

In [42]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(
    class_weight='balanced',
    max_depth=8,           # ← ОГРАНИЧЬ глубину
    min_samples_split=20,  # ← ДОБАВЬ
    min_samples_leaf=10,   # ← ДОБАВЬ
    random_state=42
)
tree.fit(X_train_processed, y_train_processed)

DecisionTreeClassifier(class_weight='balanced', max_depth=8,
                       min_samples_leaf=10, min_samples_split=20,
                       random_state=42)

## Шаг 4. Случайный лес

Воспользуйтесь `sklearn.ensemble import RandomForestClassifier`. Сравните с бейзлайном.

In [43]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(
    class_weight='balanced',
    n_estimators=200,      # ← УВЕЛИЧЬ с 100 до 200
    max_depth=12,          # ← ОГРАНИЧЬ
    min_samples_split=15,  # ← ДОБАВЬ
    min_samples_leaf=7,    # ← ДОБАВЬ
    max_features='sqrt',   # ← ДОБАВЬ
    random_state=42,
    n_jobs=-1
)
clf.fit(X_train_processed, y_train_processed)

RandomForestClassifier(class_weight='balanced', max_depth=12,
                       min_samples_leaf=7, min_samples_split=15,
                       n_estimators=200, n_jobs=-1, random_state=42)

## Шаг 5. SVM

Воспользуйтесь `sklearn.svm import SVC`. Сравните с бейзлайном.

In [44]:
from sklearn.svm import SVC

svm = SVC(
    class_weight='balanced',
    C=0.5,                 # ← УМЕНЬШИ C (сильнее регуляризация)
    kernel='rbf',
    gamma='scale',
    probability=True,
    cache_size=1000,
    random_state=42
)

svm.fit(X_train_processed, y_train_processed)

SVC(C=0.5, cache_size=1000, class_weight='balanced', probability=True,
    random_state=42)

In [62]:
models_list = []

models_list.append(("LogReg", logreg, X_test_base, y_test_base))
models_list.append(("KNN", neigh, X_test_processed, y_test_processed))
models_list.append(("Tree", tree, X_test_processed, y_test_processed))
models_list.append(("Forest", clf, X_test_processed, y_test_processed))
models_list.append(("SVM", svm, X_test_processed, y_test_processed))


In [63]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("=" * 80)
print("🎯 ПОЛНЫЙ АНАЛИЗ МОДЕЛЕЙ ПО РУЧНЫМ МЕТРИКАМ")
print("=" * 80)

# Создаём таблицу для детальных результатов
detailed_results = []

for model_name, model, X_test, y_test in models_list:
    print(f"\n{'='*50}")
    print(f"📊 МОДЕЛЬ: {model_name}")
    print('='*50)

    # 1. Предсказания
    y_pred = model.predict(X_test)

    # 2. Все твои ручные метрики
    accuracy = accuracy_manual(y_test, y_pred)
    precision = precision_manual(y_test, y_pred)
    recall = recall_manual(y_test, y_pred)
    f1 = f1_manual(y_test, y_pred)

    # 3. Выводим
    print("📈 ОСНОВНЫЕ МЕТРИКИ:")
    print(f"   Accuracy:  {accuracy:.4f}")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall:    {recall:.4f}")
    print(f"   F1-Score:  {f1:.4f}")

    # 4. ROC-AUC и PR кривые (если есть вероятности)
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]

        # ROC кривая
        fpr, tpr, _ = roc_curve_manual(y_test, y_score)
        roc_auc = roc_auc_manual(fpr, tpr)

        # PR кривая
        precision_vals, recall_vals, _ = precision_recall_curve_manual(y_test, y_score)
        avg_precision = np.trapezoid(precision_vals, recall_vals)

        print("📊 ДОПОЛНИТЕЛЬНЫЕ МЕТРИКИ:")
        print(f"   ROC-AUC:       {roc_auc:.4f}")
        print(f"   Avg Precision: {avg_precision:.4f}")

        # Сохраняем для графиков
        roc_data = (fpr, tpr, roc_auc)
        pr_data = (precision_vals, recall_vals, avg_precision)
    else:
        print("⚠️  Нет вероятностей для ROC-AUC")
        roc_auc = None
        avg_precision = None
        roc_data = None
        pr_data = None

    # 5. Анализ ошибок
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_test, y_pred)

    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        print("🔍 МАТРИЦА ОШИБОК:")
        print(f"   True Negative:  {tn}")
        print(f"   False Positive: {fp}")
        print(f"   False Negative: {fn}")
        print(f"   True Positive:  {tp}")

    # 6. Сохраняем результаты
    detailed_results.append({
        'Модель': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc if roc_auc is not None else 'N/A',
        'Avg Precision': avg_precision if avg_precision is not None else 'N/A',
        'Тест размер': len(X_test),
        'Модель тип': type(model).__name__
    })

# Создаём итоговую таблицу
results_df = pd.DataFrame(detailed_results)

print("\n" + "=" * 80)
print("🏆 ИТОГОВАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("=" * 80)
print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

🎯 ПОЛНЫЙ АНАЛИЗ МОДЕЛЕЙ ПО РУЧНЫМ МЕТРИКАМ

📊 МОДЕЛЬ: LogReg
📈 ОСНОВНЫЕ МЕТРИКИ:
   Accuracy:  0.7702
   Precision: 0.0524
   Recall:    0.2991
   F1-Score:  0.0892
📊 ДОПОЛНИТЕЛЬНЫЕ МЕТРИКИ:
   ROC-AUC:       0.5874
   Avg Precision: -0.0583
🔍 МАТРИЦА ОШИБОК:
   True Negative:  4518
   False Positive: 1211
   False Negative: 157
   True Positive:  67

📊 МОДЕЛЬ: KNN
📈 ОСНОВНЫЕ МЕТРИКИ:
   Accuracy:  0.9624
   Precision: 0.0000
   Recall:    0.0000
   F1-Score:  0.0000
📊 ДОПОЛНИТЕЛЬНЫЕ МЕТРИКИ:
   ROC-AUC:       0.5362
   Avg Precision: -0.0461
🔍 МАТРИЦА ОШИБОК:
   True Negative:  5729
   False Positive: 0
   False Negative: 224
   True Positive:  0

📊 МОДЕЛЬ: Tree
📈 ОСНОВНЫЕ МЕТРИКИ:
   Accuracy:  0.6716
   Precision: 0.0480
   Recall:    0.4107
   F1-Score:  0.0860
📊 ДОПОЛНИТЕЛЬНЫЕ МЕТРИКИ:
   ROC-AUC:       0.5698
   Avg Precision: -0.0510
🔍 МАТРИЦА ОШИБОК:
   True Negative:  3906
   False Positive: 1823
   False Negative: 132
   True Positive:  92

📊 МОДЕЛЬ: Forest
📈 ОСНОВНЫЕ МЕТРИКИ

# Задание 4. Ансамблирование

Воспользуйтесь модулем `sklearn.ensemble` для реализации ансамблирования ранее обученных моделей.
Например, вы можете использовать `sklearn.ensemble.VotingClassifier`.
 Сравните с бейзлайном.

In [64]:
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

print("=" * 70)
print("🤝 СОЗДАНИЕ VOTINGCLASSIFIER ИЗ ТВОИХ МОДЕЛЕЙ")
print("=" * 70)

# 1. Создаём VotingClassifier из твоих 3 лучших моделей
# (берём те, у которых есть predict_proba для soft voting)
voting_clf = VotingClassifier(
    estimators=[
        ('logreg', logreg),
        ('tree', tree),
        ('svm', svm)  # если у svm есть probability=True
    ],
    voting='soft',      # 'soft' для вероятностей, 'hard' для голосования
    n_jobs=-1,
    verbose=1
)

# 2. Обучаем на подвыборке для скорости
print("Обучаю VotingClassifier...")
voting_clf.fit(X_train_processed[:5000], y_train_processed[:5000])

# 3. Предсказания и оценка
y_pred_voting = voting_clf.predict(X_test_processed)
y_pred_baseline = logreg.predict(X_test_base)

# 4. Сравнение с бейзлайном (Logistic Regression)
print("\n" + "=" * 70)
print("📊 СРАВНЕНИЕ VOTINGCLASSIFIER С БЕЙЗЛАЙНОМ")
print("=" * 70)

# Метрики для VotingClassifier
acc_voting = accuracy_score(y_test_processed, y_pred_voting)
f1_voting = f1_score(y_test_processed, y_pred_voting, zero_division=0)

# Метрики для бейзлайна (LogReg)
acc_baseline = accuracy_score(y_test_base, y_pred_baseline)
f1_baseline = f1_score(y_test_base, y_pred_baseline, zero_division=0)

print(f"{'Метрика':<15} {'VotingClassifier':<20} {'LogReg (бейзлайн)':<20} {'Разница':<10}")
print("-" * 65)
print(f"{'Accuracy':<15} {acc_voting:<20.4f} {acc_baseline:<20.4f} {acc_voting - acc_baseline:+.4f}")
print(f"{'F1-Score':<15} {f1_voting:<20.4f} {f1_baseline:<20.4f} {f1_voting - f1_baseline:+.4f}")

# 5. Анализ результатов
print("\n" + "=" * 70)
print("📈 АНАЛИЗ РЕЗУЛЬТАТОВ")
print("=" * 70)

if acc_voting > acc_baseline:
    print(f"✅ VotingClassifier лучше по Accuracy на {acc_voting - acc_baseline:+.4f}")
else:
    print(f"⚠️  Бейзлайн лучше по Accuracy на {acc_baseline - acc_voting:.4f}")

if f1_voting > f1_baseline:
    print(f"✅ VotingClassifier лучше по F1-Score на {f1_voting - f1_baseline:+.4f}")
else:
    print(f"⚠️  Бейзлайн лучше по F1-Score на {f1_baseline - f1_voting:.4f}")

# 6. Матрица ошибок для VotingClassifier
print("\n🔍 МАТРИЦА ОШИБОК VOTINGCLASSIFIER:")
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test_processed, y_pred_voting)
tn, fp, fn, tp = cm.ravel()

print(f"True Negative:  {tn}")
print(f"False Positive: {fp}")
print(f"False Negative: {fn}")
print(f"True Positive:  {tp}")

🤝 СОЗДАНИЕ VOTINGCLASSIFIER ИЗ ТВОИХ МОДЕЛЕЙ
Обучаю VotingClassifier...

📊 СРАВНЕНИЕ VOTINGCLASSIFIER С БЕЙЗЛАЙНОМ
Метрика         VotingClassifier     LogReg (бейзлайн)    Разница   
-----------------------------------------------------------------
Accuracy        0.9370               0.7702               +0.1668
F1-Score        0.0554               0.0892               -0.0338

📈 АНАЛИЗ РЕЗУЛЬТАТОВ
✅ VotingClassifier лучше по Accuracy на +0.1668
⚠️  Бейзлайн лучше по F1-Score на 0.0338

🔍 МАТРИЦА ОШИБОК VOTINGCLASSIFIER:
True Negative:  5567
False Positive: 162
False Negative: 213
True Positive:  11


# Дополнительное задание**

Реализуйте в .py модуле (или в нескольких модулях, архитектура остается на ваше усматрение, будьте готовы ее объяснить):
- пайплайн обработки данных (как в задании 1),
- пайплайн обучения/дообучения ансамбля (как в заданиях 2,3). Также вычислите метрики и выведите их в консоль
- пайплайн для классификации нового объекта ансамблем моделей (без обучения этих моделей, только предсказание метрик)

